In [4]:
import re
import networkx as nx
import pickle

In [9]:
pattern_dash = r'(\w+)_HUMAN'
def gene_symbol_extractor(text, pattern:str):
        # ^ ensures start at the beginning, $ ensures end at the ')'
        match = re.search(pattern, text)
        if match:
            target = match.group(1)
            return target.upper()
        return None

In [11]:
def load_graph(filepath="../datasets/base_kgs/ppi_hc.pkl"):
    with open(filepath,'rb') as f:
        ppikg = pickle.load(f)
    print(f"Load Graph with {ppikg.number_of_nodes()} nodes and {ppikg.number_of_edges()} edges.")
    return ppikg

In [13]:
ppikg = load_graph()
for node in ppikg.nodes():
    print(node)
    gene_symbol = gene_symbol_extractor(node, pattern_dash)
    print(gene_symbol)
    break
    

Load Graph with 17042 nodes and 382526 edges.
AL1A1_HUMAN
AL1A1


### Convert Networkx Obj to PyKEEN Obj

In [24]:
import networkx as nx
import pandas as pd
from pykeen.triples import TriplesFactory
from pykeen.hpo import hpo_pipeline_from_path

In [20]:
G = load_graph("../datasets/Prime_KGs/G_adni_ADKG_all.pkl")
triples = []

# Iterate over all edges in the MultiDiGraph
for u, v, rel, data in G.edges(data=True, keys=True):
    # print(u,v,rel)
    # print(data)
    if isinstance(rel, str):
        relation = rel
    else:
        relation = data.get('type',None)
        if relation == None: relation = data.get('relation')
    if 'rev' not in relation:
        triples.append((u, relation, v))

Load Graph with 5364 nodes and 751002 edges.


In [21]:
# Convert to a DataFrame for easier handling
df = pd.DataFrame(triples, columns=['head', 'relation', 'tail'])

# Create the TriplesFactory from the pandas DataFrame
triples_factory = TriplesFactory.from_labeled_triples(
    triples=df.values,
    create_inverse_triples=True # the graph already has reverse edges
)

print(f"Number of entities: {triples_factory.num_entities}")
print(f"Number of relations: {triples_factory.num_relations}")

Number of entities: 5364
Number of relations: 28


In [25]:
# split train, val, test data
ratios = [0.8, 0.1, 0.1]
train, test, val = triples_factory.split(ratios, random_state=42)

# running hpo
config_path = "../PyKeen/configs/TransE_config_hpo.json"
transE_result = hpo_pipeline_from_path(
    config_path, 
    training = train, 
    testing=test, 
    validation=val
    )
transE_result.save_to_directory('../results/PyKeen/TransE_hpo_results')

[I 2026-05-26 00:16:19,308] A new study created in memory with name: no-name-02852792-e72f-4464-8284-85eb96c3c72b
/opt/anaconda3/envs/firegnn/lib/python3.12/site-packages/pykeen/hpo/hpo.py:970: UserWarning: The distribution is specified by [0.5, 10] and step=1.0, but the range is not divisible by `step`. It will be replaced with [0.5, 9.5].
  _kwargs[name] = trial.suggest_float(
No random seed is specified. Setting to 192019185.
No cuda devices were available. The model runs on CPU
INFO:pykeen.triples.triples_factory:Creating inverse triples.
Training epochs on cpu:   2%|▏         | 18/1000 [24:31<22:18:24, 81.78s/epoch, loss=1.34, prev_loss=1.35]
[W 2026-05-26 00:40:52,458] Trial 0 failed with parameters: {'model.embedding_dim': 256, 'model.scoring_fct_norm': 1, 'loss.margin': 9.5, 'optimizer.lr': 0.07118482561320064, 'negative_sampler.num_negs_per_pos': 28, 'training.batch_size': 256} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/opt

KeyboardInterrupt: 